# Desafio - RPG de mesa ⚔

Esse ano tivemos o fim de uma das séries mais icônicas da história: Stranger Things. Para homenageá-la, teremos um desafio temático de **RPG de mesa**.

## Contexto

Quando estamos jogando RPG de mesa (como Dungeons and Dragons, por exemplo), temos o famoso momento do combate: em algum momento da história os jogadores entram em um combate com inimigos, tendo que utilizar ataques e habilidades seus personagens para eliminá-los. 

Cada personagem possui atributos diferentes, como inteligência, força, destreza, vitalidade, etc, além de magias e armas. Assim, cada personagem é capaz de ter algum tipo de ação sobre os inimigos, com mais ou menos dano em seus pontos de vida. Sabendo como cada personagem funciona, é possível criar diversas estratégias para cada cenário de batalha

## Objetivo

Nosso objetivo será criar um **simulador simplificado de combate**, utilizando estruturas de dados clássicas.

Teremos **personagens** com os seguintes atributos:
- Nome
- Tipo (jogador/monstro)
- Classe (guerreiro/mago/druida/etc para jogador, orc/beholder/etc para monstro)
- Dano de Ataque
- Pontos de Vida totais
- Pontos de Vida atuais
- Habilidade

Além disso, os personagens também podem ter **habilidades**. Essas habilidades serão usadas opcionalmente durante os turnos no combate, de maneira aleatória.
Uma habilidade tem os seguintes atributos:
- Nome
- Modificador

Se um personagem usa uma **habilidade**, o seu dano de ataque é multiplicado pelo modificador desta **habilidade**.

**Exemplo:**

```
Personagem 1
-------------------
Nome: Aragorn
Classe: Guerreiro
Dano de ataque: 10
Pontos de vida totais: 100
Pontos de vida atuais: 50
Habilidade:
    Nome: Fireball
    Modificador: 1.2
```

Se `Aragorn` realizar um ataque utilizando a **habilidade** `Fireball`, seu ataque será `12`, porque será multiplicado pelo modificador.

---

Essa é uma simplificação do nosso problema, para podermos representar tanto os jogadores quanto os inimigos (usando monstro como classe).

## Entendendo o combate

Para iniciar um combate, precisamos saber a sequência de turnos que ele executará. Isso é definido pela **iniciativa**. 
Ao entrar em combate, todos os jogadores rolam seus dados para determinar a sua ordem de atuação.

Imagine o seguinte cenário:

- Jogador A
- Jogador B
- Jogador C
- Monstro 1
- Monstro 2

Cada um dos jogadores e monstros (interpretados pelo mestre) rolam os dados para determinar sua sequência de atuação. 

- Jogador A (rolou `20`)
- Jogador B (rolou `10`)
- Jogador C (rolou `5`)
- Monstro 1 (rolou `7`)
- Monstro 2 (rolou `1`)

Neste exemplo, a ordem fica:

- Monstro 2
- Jogador C
- Monstro 1
- Jogador B
- Jogador A

Ou seja, os itens são **ordenados** de acordo com sua rolagem de iniciativa.

## Estruturando nosso projeto

Considerando tudo o que falamos, precisamos definir quais estruturas teremos no nosso simulador.

Um caminho é começar pelo mapeamento de nossas estruturas de dados específicas:

```c
#define JOGADOR 1
#define MONSTRO 2

#define MAX_NOME 50
#define MAX_CLASSE 50
#define MAX_HABILIDADE 50

typedef struct habilidade {
    char nome[MAX_HABILIDADE];
    float modificador;
} Habilidade;

typedef struct personagem {
    char nome[MAX_NOME];
    int tipo; // JOGADOR ou MONSTRO
    char classe[MAX_CLASSE];
    int vida;
    int dano;
    int iniciativa;
    int temHabilidade;
    Habilidade habilidade;
} Personagem;
```

Para construir o simulador, utilizaremos uma lista circular para determinar os turnos dos personagens/monstros.
Assim, teremos que ter as seguintes estruturas de dados:

```c
typedef struct no {
    Personagem personagem;
    struct no *prox;
} No;

typedef No *Celula;

typedef struct lista {
    Celula inicio;
    Celula fim;
} Lista;

typedef Lista *ListaCircular;

ListaCircular novaListaCircular(void) {
    ListaCircular lista = (ListaCircular) malloc(sizeof(Lista));
    if (lista == NULL) return NULL;
    lista->inicio = NULL;
    lista->fim = NULL;
    return lista;
}

Celula novaCelula(Personagem p) {
    Celula nova = (Celula) malloc(sizeof(No));
    if (nova == NULL) return NULL;
    nova->personagem = p;
    nova->prox = NULL;
    return nova;
}
```

Dessa maneira, tudo o que precisamos é conseguir realizar as operações de lista ligada circular, utilizando-as para nosso simulador de combate!

## O que seu programa deve ser capaz de executar

O seu simulador deverá ser capaz de:
- Rolar iniciativa para cada jogador registrado na lista, determinando um valor aleatório entre 1 e 20
- Garantir que os turnos estejam na ordem determinada pela iniciativa
- Para cada turno de um personagem:
  - O personagem atual ataca o personagem mais próximo possível (jogadores atacam monstros e monstros jogadores), usando seu dano de ataque, alterando os pontos de vida do inimigo. Existe uma probabilidade de 10% do personagem usar sua habilidade. Se cair nessa opção, o ataque é multiplicado pelo modificador da habilidade.
  - No caso de um personagem da lista ter seus pontos de vida zerados, ele deve ser removido da lista
- Ao final, o simulador deve mostrar quem venceu: os jogadores ou os monstros.

É importante que o seu simulador mostre o que está acontecendo a cada turno:
- De quem é o turno (com seus pontos de vida atuais/totais)
- Quem está atacando quem?
- Foi usada a habilidade? Se sim, mostre o nome e o modificador
- Quanto de dano foi causado?

Para facilitar, já deixei um esqueleto do programa com funções a serem implementadas. Depois, se você gostar muito da ideia, pode evoluir e fazer suas próprias mudanças no simulador, para deixá-lo mais interessante :).

## Esqueleto para implementação

Esse é um esqueleto com tudo que é necessário para que seu simulador funcione.
Deixei algumas funções já implementadas, para te ajudar ;).

Se você conseguir implementar as funções pendentes, já terá como ver as saídas e brincar com seu simulador!

**Atenção: o que deve ser commitado em seu repositório público é jsutamente o conteúdo da célula abaixo, com suas implementações**

```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define JOGADOR 1
#define MONSTRO 2

#define MAX_NOME 50
#define MAX_CLASSE 50
#define MAX_HABILIDADE 50

typedef struct habilidade {
    char nome[MAX_HABILIDADE];
    float modificador;
} Habilidade;

typedef struct personagem {
    char nome[MAX_NOME];
    int tipo; // JOGADOR ou MONSTRO
    char classe[MAX_CLASSE];
    int vida;
    int dano;
    int iniciativa;
    int temHabilidade;
    Habilidade habilidade;
} Personagem;

typedef struct no {
    Personagem personagem;
    struct no *prox;
} No;

typedef No *Celula;

typedef struct lista {
    Celula inicio;
    Celula fim;
} Lista;

typedef Lista *ListaCircular;

/* =========================================================
   FUNCOES PRONTAS
   ========================================================= */

ListaCircular novaListaCircular(void) {
    ListaCircular lista = (ListaCircular) malloc(sizeof(Lista));
    if (lista == NULL) return NULL;
    lista->inicio = NULL;
    lista->fim = NULL;
    return lista;
}

Celula novaCelula(Personagem p) {
    Celula nova = (Celula) malloc(sizeof(No));
    if (nova == NULL) return NULL;
    nova->personagem = p;
    nova->prox = NULL;
    return nova;
}

int listaVazia(ListaCircular lista) {
    return lista == NULL || lista->inicio == NULL;
}

void insereNoFimCircular(ListaCircular lista, Personagem p) {
    Celula nova = novaCelula(p);
    if (lista == NULL || nova == NULL) return;

    if (lista->inicio == NULL) {
        lista->inicio = nova;
        lista->fim = nova;
        nova->prox = nova;
    } else {
        nova->prox = lista->inicio;
        lista->fim->prox = nova;
        lista->fim = nova;
    }
}

/* Ordem crescente de iniciativa, como no enunciado da aula */
void insereOrdenadoPorIniciativa(ListaCircular lista, Personagem p) {
    Celula nova, aux;

    if (lista == NULL) return;

    nova = novaCelula(p);
    if (nova == NULL) return;

    if (lista->inicio == NULL) {
        lista->inicio = nova;
        lista->fim = nova;
        nova->prox = nova;
        return;
    }

    if (p.iniciativa < lista->inicio->personagem.iniciativa) {
        nova->prox = lista->inicio;
        lista->inicio = nova;
        lista->fim->prox = lista->inicio;
        return;
    }

    if (p.iniciativa >= lista->fim->personagem.iniciativa) {
        nova->prox = lista->inicio;
        lista->fim->prox = nova;
        lista->fim = nova;
        return;
    }

    aux = lista->inicio;
    while (aux->prox != lista->inicio &&
           aux->prox->personagem.iniciativa <= p.iniciativa) {
        aux = aux->prox;
    }

    nova->prox = aux->prox;
    aux->prox = nova;
}

void printListaCircular(ListaCircular lista) {
    Celula aux;

    if (listaVazia(lista)) {
        printf("[lista vazia]\n");
        return;
    }

    aux = lista->inicio;
    do {
        printf("%s{%s, Classe=%s, HP=%d, D=%d, Ini=%d}",
               aux->personagem.nome,
               aux->personagem.tipo == JOGADOR ? "Jogador" : "Monstro",
               aux->personagem.classe,
               aux->personagem.vida,
               aux->personagem.dano,
               aux->personagem.iniciativa);
        if (aux->personagem.temHabilidade) {
            printf("[Hab=%s x%.1f]",
                   aux->personagem.habilidade.nome,
                   aux->personagem.habilidade.modificador);
        }
        aux = aux->prox;
        if (aux != lista->inicio) printf(" -> ");
    } while (aux != lista->inicio);
    printf("\n");
}

int contarTipo(ListaCircular lista, int tipo) {
    Celula aux;
    int total = 0;

    if (listaVazia(lista)) return 0;

    aux = lista->inicio;
    do {
        if (aux->personagem.tipo == tipo) total++;
        aux = aux->prox;
    } while (aux != lista->inicio);

    return total;
}

void liberarLista(ListaCircular lista) {
    Celula atual, prox;

    if (lista == NULL) return;
    if (lista->inicio == NULL) {
        free(lista);
        return;
    }

    atual = lista->inicio->prox;
    while (atual != lista->inicio) {
        prox = atual->prox;
        free(atual);
        atual = prox;
    }

    free(lista->inicio);
    free(lista);
}

/* =========================================================
   FUNCOES PARA IMPLEMENTAR
   ========================================================= */

/*
TODO 1:
Implemente uma funcao que, a partir do personagem atual, percorra a lista
circular a partir do proximo no e devolva o PRIMEIRO inimigo encontrado.
Se o atual for jogador, deve procurar um monstro.
Se o atual for monstro, deve procurar um jogador.
Se nao existir inimigo, retorne NULL.
*/
Celula buscaInimigoMaisProximo(Celula atual) {
    /* IMPLEMENTAR */
    return NULL;
}

/*
TODO 2:
Implemente a remocao de um no especifico de uma lista circular.
Casos importantes:
- lista vazia
- alvo eh o unico elemento
- alvo eh o inicio
- alvo eh o fim
- alvo esta no meio
*/
void removeDaListaCircular(ListaCircular lista, Celula alvo) {
    /* IMPLEMENTAR */
}

/*
TODO 3:
Implemente UM turno de combate.
Passos sugeridos:
1. encontrar o inimigo mais proximo
2. guardar quem sera o proximo da rodada
3. calcular o dano base do personagem atual
4. se ele tiver habilidade, gerar um numero aleatorio entre 0 e 1
   e, com 20%% de chance, multiplicar o dano pelo modificador
5. aplicar o dano ao alvo
6. se o alvo morrer, removê-lo da lista
7. retornar o ponteiro do proximo personagem que devera agir
*/
Celula executaUmTurno(ListaCircular lista, Celula atual) {
    /* IMPLEMENTAR */
    return NULL;
}

/* =========================================================
   TESTES / MAINS DE EXEMPLO
   ========================================================= */

void teste1_iniciativa(void) {
    ListaCircular lista = novaListaCircular();

    Personagem a  = {"Jogador A", JOGADOR, "Guerreiro", 30,  8, 20, 1, {"Ataque Heroico", 1.5f}};
    Personagem b  = {"Jogador B", JOGADOR, "Mago",      20, 10, 10, 1, {"Bola de Fogo",   2.0f}};
    Personagem c  = {"Jogador C", JOGADOR, "Ladino",    25,  6,  5, 0, {"",               0.0f}};
    Personagem m1 = {"Monstro 1", MONSTRO, "Orc",       18,  7,  7, 0, {"",               0.0f}};
    Personagem m2 = {"Monstro 2", MONSTRO, "Goblin",    15,  4,  1, 0, {"",               0.0f}};

    insereOrdenadoPorIniciativa(lista, a);
    insereOrdenadoPorIniciativa(lista, b);
    insereOrdenadoPorIniciativa(lista, c);
    insereOrdenadoPorIniciativa(lista, m1);
    insereOrdenadoPorIniciativa(lista, m2);

    printf("=== TESTE 1: ORDEM DE INICIATIVA ===\n");
    printListaCircular(lista);

    liberarLista(lista);
}

void teste2_um_turno_sem_morte(void) {
    ListaCircular lista = novaListaCircular();

    Personagem a = {"Aragorn", JOGADOR, "Guerreiro", 30, 10, 4, 0, {"", 0.0f}};
    Personagem g = {"Gandalf", JOGADOR, "Mago",      20, 12, 8, 0, {"", 0.0f}};
    Personagem o = {"Orc",     MONSTRO, "Orc",       18,  5, 6, 0, {"", 0.0f}};

    insereOrdenadoPorIniciativa(lista, a);
    insereOrdenadoPorIniciativa(lista, g);
    insereOrdenadoPorIniciativa(lista, o);

    printf("=== TESTE 2: UM TURNO SEM MORTE ===\n");
    printf("Antes do turno:\n");
    printListaCircular(lista);

    executaUmTurno(lista, lista->inicio);

    printf("Depois do turno:\n");
    printListaCircular(lista);

    liberarLista(lista);
}

void teste3_remocao_apos_derrota(void) {
    ListaCircular lista = novaListaCircular();
    Celula atual;

    Personagem a = {"Aragorn", JOGADOR, "Guerreiro", 30, 20, 4, 0, {"", 0.0f}};
    Personagem g = {"Gandalf", JOGADOR, "Mago",      20, 12, 8, 0, {"", 0.0f}};
    Personagem o = {"Orc",     MONSTRO, "Orc",       15,  5, 6, 0, {"", 0.0f}};

    insereOrdenadoPorIniciativa(lista, a);
    insereOrdenadoPorIniciativa(lista, g);
    insereOrdenadoPorIniciativa(lista, o);

    printf("=== TESTE 3: REMOCAO APOS DERROTA ===\n");
    printf("Antes do turno:\n");
    printListaCircular(lista);

    atual = lista->inicio;
    executaUmTurno(lista, atual);

    printf("Depois do turno:\n");
    printListaCircular(lista);

    liberarLista(lista);
}

void teste4_varios_turnos_com_habilidade(void) {
    ListaCircular lista = novaListaCircular();
    Celula atual;
    int turno;

    Personagem a  = {"Jogador A", JOGADOR, "Guerreiro", 25,  8, 2, 1, {"Golpe Heroico", 1.5f}};
    Personagem b  = {"Jogador B", JOGADOR, "Mago",      18, 10, 5, 1, {"Raio Arcano",   2.0f}};
    Personagem m1 = {"Monstro 1", MONSTRO, "Orc",       16,  6, 3, 0, {"",              0.0f}};
    Personagem m2 = {"Monstro 2", MONSTRO, "Goblin",    12,  4, 1, 0, {"",              0.0f}};

    insereOrdenadoPorIniciativa(lista, a);
    insereOrdenadoPorIniciativa(lista, b);
    insereOrdenadoPorIniciativa(lista, m1);
    insereOrdenadoPorIniciativa(lista, m2);

    atual = lista->inicio;

    /* Seed fixa para deixar a execucao reproduzivel */
    srand(42);

    printf("=== TESTE 4: VARIOS TURNOS COM HABILIDADE ===\n");
    printListaCircular(lista);

    for (turno = 1; turno <= 6; turno++) {
        if (contarTipo(lista, JOGADOR) == 0 || contarTipo(lista, MONSTRO) == 0) {
            break;
        }

        printf("\nTurno %d\n", turno);
        atual = executaUmTurno(lista, atual);
        printListaCircular(lista);
    }

    liberarLista(lista);
}

int main(void) {
    /*
    Descomente um teste por vez, ou deixe todos ativos apos implementar
    as funcoes pedidas.
    */

    teste1_iniciativa();
    printf("\n");

    teste2_um_turno_sem_morte();
    printf("\n");

    teste3_remocao_apos_derrota();
    printf("\n");

    teste4_varios_turnos_com_habilidade();

    return 0;
}
```

A saída para estes testes é algo parecido com isso:

```
=== TESTE 1: ORDEM DE INICIATIVA ===
Monstro 2{Monstro, Classe=Goblin, HP=15, D=4, Ini=1} -> Jogador C{Jogador, Classe=Ladino, HP=25, D=6, Ini=5} -> Monstro 1{Monstro, Classe=Orc, HP=18, D=7, Ini=7} -> Jogador B{Jogador, Classe=Mago, HP=20, D=10, Ini=10}[Hab=Bola de Fogo x2.0] -> Jogador A{Jogador, Classe=Guerreiro, HP=30, D=8, Ini=20}[Hab=Ataque Heroico x1.5]

=== TESTE 2: UM TURNO SEM MORTE ===
Antes do turno:
Aragorn{Jogador, Classe=Guerreiro, HP=30, D=10, Ini=4} -> Orc{Monstro, Classe=Orc, HP=18, D=5, Ini=6} -> Gandalf{Jogador, Classe=Mago, HP=20, D=12, Ini=8}
Aragorn ataca Orc causando 10 de dano.
Depois do turno:
Aragorn{Jogador, Classe=Guerreiro, HP=30, D=10, Ini=4} -> Orc{Monstro, Classe=Orc, HP=8, D=5, Ini=6} -> Gandalf{Jogador, Classe=Mago, HP=20, D=12, Ini=8}

=== TESTE 3: REMOCAO APOS DERROTA ===
Antes do turno:
Aragorn{Jogador, Classe=Guerreiro, HP=30, D=20, Ini=4} -> Orc{Monstro, Classe=Orc, HP=15, D=5, Ini=6} -> Gandalf{Jogador, Classe=Mago, HP=20, D=12, Ini=8}
Aragorn ataca Orc causando 20 de dano.
Orc foi derrotado!
Depois do turno:
Aragorn{Jogador, Classe=Guerreiro, HP=30, D=20, Ini=4} -> Gandalf{Jogador, Classe=Mago, HP=20, D=12, Ini=8}

=== TESTE 4: VARIOS TURNOS COM HABILIDADE ===
Monstro 2{Monstro, Classe=Goblin, HP=12, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=25, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=16, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=18, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 1
Monstro 2 ataca Jogador A causando 4 de dano.
Monstro 2{Monstro, Classe=Goblin, HP=12, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=21, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=16, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=18, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 2
Jogador A usa habilidade Golpe Heroico!
Jogador A ataca Monstro 1 causando 12 de dano.
Monstro 2{Monstro, Classe=Goblin, HP=12, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=21, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=4, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=18, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 3
Monstro 1 ataca Jogador B causando 6 de dano.
Monstro 2{Monstro, Classe=Goblin, HP=12, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=21, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=4, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=12, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 4
Jogador B ataca Monstro 2 causando 10 de dano.
Monstro 2{Monstro, Classe=Goblin, HP=2, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=21, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=4, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=12, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 5
Monstro 2 ataca Jogador A causando 4 de dano.
Monstro 2{Monstro, Classe=Goblin, HP=2, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=17, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Monstro 1{Monstro, Classe=Orc, HP=4, D=6, Ini=3} -> Jogador B{Jogador, Classe=Mago, HP=12, D=10, Ini=5}[Hab=Raio Arcano x2.0]

Turno 6
Jogador A ataca Monstro 1 causando 8 de dano.
Monstro 1 foi derrotado!
Monstro 2{Monstro, Classe=Goblin, HP=2, D=4, Ini=1} -> Jogador A{Jogador, Classe=Guerreiro, HP=17, D=8, Ini=2}[Hab=Golpe Heroico x1.5] -> Jogador B{Jogador, Classe=Mago, HP=12, D=10, Ini=5}[Hab=Raio Arcano x2.0]
```